# Stochastic Variational Inference

Source:<P>
    https://andrewcharlesjones.github.io/journal/svi.html

Adapted:

    Antonio Esteves @ UMinho, Mar 2024

## Stochastic variational inference
 
Variational inference (VI) is a framework for approximating intractable posterior distributions. In some respects, VI converts an inference problem into an optimization problem. Stochastic variational inference (SVI) is a family of methods that exploits stochastic optimization techniques to speed up variational approaches and scale them to large datasets. Here, we review SVI as introduced by [Hoffman et al](https://www.jmlr.org/papers/volume14/hoffman13a/hoffman13a.pdf).

We begin by reviewing the modeling setup followed by Hoffman et al. Then, we describe SVI and its implementation.

## The model

Although the fundamental ideas of SVI can be applied to arbitrary probabilistic models, the paper by Hoffman et al. focuses on a subset of models that are commonly used in practice. We review these modeling assumptions below.

### Global and local variables

To start, we assume we have $n$ data points $\{x_1,\dots,x_n\}$ with $x_i \in \mathbb{R}^p$. The model assumes two distinct types of latent variables: local latent variables $\{z_1,\dots,z_n\}$, with $z_i \in \mathbb{R}^k$ that correspond to individual data points, and global latent variables $\beta$ that describe all of the data points. Finally, we have a set of fixed hyperparameters $\alpha$ which we assume to be constant and specified by the modeler. The joint likelihood of the model is

\begin{equation}
p(x, z, \beta ; \alpha) = p(\beta ; \alpha) \prod\limits_{i=1}^n p(x_i, z_i | \beta)
\end{equation}

where we use $x$ and $z$ to denote the full data and local latent variables. For simplicity, the model assumes that only the global latent variables $\beta$ depend on the fixed hyperparameters $\alpha$. Note that the local latent variables are independent of one another.

Presented with some data, we would like to compute the posterior distribution. In this generic model, the posterior is given by

\begin{equation}
p(z, \beta | x ; \alpha) = \frac{p(x, z, \beta ; \alpha)}{p(x)} = \frac{p(x | z, \beta, \alpha) p(z, \beta | \alpha)}{p(x)}
\end{equation}

where $z$ and $x$ represent all the local latent variables and data points. Unfortunately, the integral in the denominator is intractable for most semi-complicated models:

\begin{equation}
p(x) = \int p(x, z, \beta ; \alpha) dz d\beta
\end{equation}

This intractability motivates the need for approximations to the posterior. Variational inference is a popular approach for formulating these approximations.

### Example: Mixture of Gaussians

As an example of the types of models that fit into this framework, consider a simple mixture of $K$  univariate Gaussians. The model is

\begin{align}
x_i \mid z_i = k, \mu_{z_i}, \sigma^2_0 &\sim \mathcal{N}(\mu_k, \sigma^2_0) \\
z_i &\sim \text{Categorical}(\theta), ~i=1,\dots,n \\ 
\mu_k &\sim \mathcal{N}(0, 1),~~~k=1,\dots,K
\end{align}

where $z_i$ indicates the mixture membership of sample $i$, $\mu_k$ is the mean of mixture component $k$, $\theta \in \Delta^K$ contains the mixture probabilities, and we assume $\sigma^2_0$ is known and shared across components.

In this example, the local latent variables $z$ are the sample-specific mixture memberships $\{z_i\}_{i=1}^n$. The global latent variables $\beta$ are the mixture means $\{\mu_k\}_{k=1}^K$ and the mixture probabilities $\theta$. The only hyperparameter $\alpha$ is the variance $\sigma^2_0$.

### Complete conditionals and the exponential family

We need one more assumption before we jump into SVI. Specifically, we assume a certain form for the complete conditional distributions. A complete conditional distribution is the distribution of a latent variable conditioned on all other latent variables and the data. In other words, it is the posterior distribution if we also condition on all other parameters/latent variables. For example, the complete conditional for $\beta$ is $p(\beta | x, z, \alpha)$, and the complete conditional for one local variable $z_{ik}$ is $p(z_{ik} | x, \beta, z_{i, -k}, \alpha)$ where $z_{i, -k}$ contains all local latent variables for data point $i$ except the $k$-th one. Recall that the local latent variables are independent of one another, so 

\begin{equation}
p(z_{ik} | x, \beta, z_{i, -k}, z_{[n] \setminus i}, \alpha) = p(z_{ik} | x, \beta, z_{i, -k}, \alpha)
\end{equation}

The primary assumption about the complete conditionals is that they are in the exponential family. To give a brief review of the exponential family, recall that a distribution $p(x)$ with parameter vector $\theta$ belongs to the exponential family if it has the following form:

\begin{equation}
p(x | \theta) = h(x) \exp\{\eta(\theta)^\top t(x) - a(\theta)\}
\end{equation}

where $h(x)$ is the base measure, $\eta(\theta)$ is the natural parameter, $t(x)$ is a sufficient statistic, and $a(\theta)$ is the base measure. The section at the end of this notebook shows how the Gaussian is expressed in the exponential notation. A nice property of the exponential family is that the gradient of the log-normalizer $a$ is equal to the expectation of the sufficient statistic $t$:

\begin{equation}
\nabla_\theta a(\theta) = \mathbb{E}[t(\theta)]
\tag{1}
\end{equation}
 
We assume the complete conditionals $p(\beta | x, z ; \alpha)$ and $p(z_{ik} | x, \beta, z_{i, -k} ; \alpha)$ can be put in this exponential family form. Introducing new notation for the exponential family form of these complete conditionals, we have

\begin{align}
p(\beta | x, z ; \alpha) &=  h(\beta) \exp\{\eta_g(x, z ; \alpha)^\top t(\beta) - a_g(x, z ; \alpha)\} \\ 
p(z_{ik} | x, \beta, z_{i, -k} ; \alpha) &= h(z_{ik}) \exp\{\eta_\ell(x, \beta, z_{i, -k} ; \alpha)^\top t(z_{ik}) - a_\ell(x, \beta, z_{i, -k} ; \alpha)\}
\end{align}

where the subscript on $\eta$ and $a$ indicates whether it applies to the global or local latent variables.

This exponential family assumption will simplify the computations. Moreover, it is a common assumption in many commonly-used models. Now that we have fully described the modeling setup, we can move onto variational inference.

## Mean-field VI

As mentioned above, VI approaches seek to approximate the posterior $p(z, \beta | x ; \alpha)$ with a surrogate distribution $q(z, \beta)$. One of the simplest forms of VI is mean-field VI, which assumes that the variational distribution factorizes into a product of individual variational distributions:

\begin{equation}
	q(z, \beta) = q(\beta | \lambda) \prod\limits_{i=1}^n q(z_i | \phi_i)
\end{equation}

where $\lambda$ and $\{\phi_i\}$ are the **variational parameters** that govern the global and local variational distributions, respectively.

The evidence lower bound (ELBO) is given by:

\begin{equation}
\mathcal{L} = \mathbb{E}_{z, \beta \sim q}\left[\log \frac{p(x, z, \beta)}{q(z, \beta)}\right] \leq \log p(x)
\end{equation}


We want to maximize ELBO with respect to the variational parameters. We start by maximizing ELBO with respect to the global variational parameters $\lambda$. Recall that maximizing ELBO is equivalent to minimizing the KL divergence between the true posterior and the variational distribution $q$.

Rewriting ELBO and ignoring the terms that do not depend on $\lambda$:

\begin{align}
\mathcal{L} &= \mathbb{E}\left[\log p(x, z, \beta) - \log q(z, \beta)\right] \\ 
&= \mathbb{E}\left[\log \left(p(\beta | x, z) p(x, z)\right) - \log \left(q(z) q(\beta)\right)\right] & \text{(probability chain rule)} \\ 
&= \mathbb{E}\left[\log p(\beta | x, z) - q(\beta)\right] + constant & \text{(constant w.r.t. $\lambda$)}
\end{align}

Now, taking the gradient, we have:

\begin{equation}
\nabla_\lambda \mathcal{L} = \mathbb{E}\left[\nabla_\lambda \log p(\beta | x, z) - \nabla_\lambda q(\beta)\right]
\end{equation}

Using our assumption that the complete conditional distributions belong to the exponential family, the gradient simplifies as follows. First, we write out the exponential family form again,

\begin{align}
\nabla_\lambda \mathcal{L} &= \mathbb{E}\left[\nabla_\lambda (\log (h(\beta) \exp\{\eta_g(x, z; \alpha)^\top T(\beta) - a_g(x, z; \alpha)\})) \right] \\ 
& - \mathbb{E}\left[\nabla_\lambda (\log (h(\beta) \exp\{\lambda^\top T(\beta) - a_g(\lambda)\})) \right]
\end{align}

Simplifying further and canceling like terms, we have:

\begin{align}
\nabla_\lambda \mathcal{L} &= \mathbb{E}\left[\nabla_\lambda ( \eta_g(x, z; \alpha)^\top T(\beta) - a_g(x, z; \alpha) \right] \\ 
&-\mathbb{E}\left[\nabla_\lambda ( \lambda^\top T(\beta) - a_g(\lambda)\right]
\end{align}

Using the exponential family property described in Equation (1), we have

\begin{align}
\nabla_\lambda \mathcal{L} &= \mathbb{E}\left[\nabla_\lambda ( \eta_g(x, z; \alpha)^\top \nabla_\lambda a_g(\lambda) - a_g(\eta_g(x, z; \alpha)) \right] \\
&-\mathbb{E}\left[\nabla_\lambda ( \lambda^\top \nabla_\lambda a_g(\lambda) - a_g(\lambda)\right]
\end{align}

Finally, we can compute the relevant gradients, yielding

\begin{align}
\nabla_\lambda \mathcal{L} &= \mathbb{E}\left[\nabla^2_\lambda a_g(\lambda) \eta_g(x, z; \alpha) + \nabla_\lambda a_g(\lambda) + \nabla^2_\lambda a_g(\lambda) \lambda - \nabla_\lambda a_g(\lambda)\right] \\
&= \nabla^2_\lambda a_g(\lambda) (\mathbb{E}[\eta_g(x, z; \alpha)] - \lambda)
\end{align}

We can see that the gradient will be zero when

\begin{equation}
\lambda = \mathbb{E}[\eta_g(x, z; \alpha)]
\tag{2}
\end{equation}
**(Global update)**

In other words, the global variational parameter that maximizes ELBO is the expectation of the natural parameter of the exponential for the complete conditional distribution $p(\beta | x, z; \alpha)$.

A very similar derivation for the local latent variables shows that, conditioned on the global latent variables and the data, ELBO is locally maximized when:

\begin{equation}
	\phi_{ik} = \mathbb{E}[\eta_\ell(x, \beta, z_{i, -k} ; \alpha)]
 \tag{3}
\end{equation}
**(Local update)**

These updates for $\lambda$ and $\phi$ can be repeated in a coordinate ascent procedure, which results in a very general algorithm for mean-field VI.

## Stochastic VI

We now focus our attention on using SVI to solve two computational issues associated with the coordinate ascent approach to the mean-field VI.

First, notice that before we begin coordinate ascent, all local and global latent variables will be initialized randomly (or possibly according to some more informed technique). If we start our ascent optimization by first optimizing the global latent variables, the locally optimal value of $\lambda$ in Equation (2) will depend on the randomly-initialized values of the local latent variables $\phi$. However, these random variables are not informative at all, which essentially amounts to a waste of computation on the first iteration.

Second, each iteration of coordinate ascent requires iterating over all of the local latent variables. For large datasets with lots of samples (many datasets now have sample sizes on the order of $10^5$, $10^6$, or higher), this becomes computationally burdensome. Although this step can be parallelized across machines due to the independence of the local latent variables, it would be even better if we could subsample the data appropriately and not require the entire dataset.

SVI solves both of these issues with a fairly simple trick from stochastic optimization. Instead of using the entire dataset on every iteration, we can randomly sample one data point, and compute the parameter updates as if this sample were our entire dataset. We expect that this procedure is equivalent to performing updates using the complete dataset.

More specifically, consider again the ELBO for our model:

\begin{equation}
\mathcal{L} = \mathbb{E}_q\left[\log \frac{p(x, z, \beta)}{q(z) q(\beta)} \right]
\end{equation}

We can write the ELBO separating the global and local terms:

\begin{equation}
\mathcal{L} = \underbrace{\mathbb{E}_q[\log p(\beta) - \log q(\beta)]}_{\text{global}} + \underbrace{\sum\limits_{i=1}^n \mathbb{E}_q[p(x_i, z_i | \beta) - q(z_i)]}_{\text{local}}
\end{equation}

Suppose we are in the middle of a coordinate ascent iteration, and we have maximized the ELBO with respect to the local variational parameters $\phi$. Then, the ELBO is a function of the global variational parameters $\lambda$, and we have:

\begin{equation}
\mathcal{L}(\lambda) = \mathbb{E}_q[\log p(\beta) - \log q(\beta)] + \sum\limits_{i=1}^n \max_{\phi_i} \mathbb{E}_q[p(x_i, z_i | \beta) - q(z_i)]
\end{equation}

We can now make a stochastic approximation to this locally-optimized ELBO. If we randomly sample a data point index $i \sim Uniform(1,n)$, we can approximate $\mathcal{L}(\lambda)$ as if this data point were the entire dataset:

\begin{equation}
\widehat{\mathcal{L}}(\lambda) = \mathbb{E}_q[\log p(\beta) - \log q(\beta)] + n \max_{\phi_i} \mathbb{E}_q[p(x_i, z_i | \beta) - q(z_i)]
\end{equation}

Note that we have multipled the local term by $n$ to simulate as if we have replicated this data point $n$ times. We then have an unbiased estimator for the ELBO:

\begin{equation}
\mathbb{E}[\widehat{\mathcal{L}}(\lambda)] = \mathcal{L}(\lambda)
\end{equation}

We can then take a stochastic, noisy gradient of this quantity, and use the Robbins Monro algorithm to optimize the ELBO.

**SVI algorithm:**

While ELBO $\mathcal{L}$ not converged:<P>
1. Sample a random data point index from the Uniform distribution: $i \sim Uniform(1,n)$
2. Update the local variational parameter corresponding to $x_i$:<P>
    $\phi_{ik}=\mathbb{E}[\eta_g(x_i,z_i)]$
3. Compute the intermediate global variational parameter: <P>
   $\widehat{\lambda} = \mathbb{E}[\eta_g(x_i, z_i)]$
4. Update the global variational parameter using a weighted combination of the updated value and the previous value:<P>
   $\lambda_t = (1 - \rho) \lambda_{t-1} + \rho \widehat{\lambda}$

## Natural gradient

The last ingredient for SVI is the natural gradient. The natural gradient is a generalization of the generic gradient that accounts for the information geometry of a parameter space.

## Example: Mixture of Gaussians

To show SVI in practice, we work through a simple example using a mixture of univariate Gaussians. Consider the following model:

\begin{align}
x_i | z_i, \mu_k \sim \mathcal{N}(\mu_k, \sigma^2_0) \\ 
z_i  \sim \text{Multinomial}(\boldsymbol{\pi}) \\ 
\mu_k \sim \mathcal{N}(0, 1)
\end{align}

where $\sigma^2_0$ is a known noise variance term, and $\boldsymbol{\pi} = [\pi_1, \dots, \pi_K]^\top$ is a vector of prior class probabilities. Here, we assume equal prior weight for each class, $\pi_1 = \cdots = \pi_K = \pi_0$.

Our latent variables of interest are $\{z_i\}_{i=1}^n$ and $\{\mu_k\}_{k=1}^K$. First, let us compute the complete conditionals. For the local latent variables, we have:

\begin{align}
p(z_i = k | x_i, \mu_k) &= \frac{1}{C_1} p(x_i | z_i = k, \mu_k) p(z_i = k) \\
&= \frac{1}{C_1} \pi_0 \mathcal{N}(x_i | \mu_k, \sigma^2_0)
\end{align}

where the normalization term is a sum over mixture components,

\begin{equation}
C_1 = \sum\limits_{k=1}^K \pi_0 \mathcal{N}(x_i | \mu_k, \sigma^2_0)
\end{equation}

This implies that the complete conditional is another multinomial distribution with natural parameter $\mathbb{E}_{q(\mu_k)}[\log p(z_i = k | x_i, \mu_k)] = \int q(\mu_k) \log \frac{1}{C_1}p(z_i = k | x_i, \mu_k) d\mu_k$, which only depends on the global latent variables. Taking the expectation with respect to the variational distribution $q$, we have

\begin{equation}
\mathbb{E}_{q(\mu_k)}[\log p(z_i = k | x_i, \mu_k)] = \int q(\mu_k) \log \frac{1}{C_1}p(z_i = k | x_i, \mu_k) d\mu_k
\end{equation}

Here, we restrict the variational variance term to be 1 for simplicity, so this reduces to

\begin{equation}
\mathbb{E}_{q(\mu_k)}[\log p(z_i = k | x_i, \mu_k)] = \log \mathcal{N}(x_i | \lambda_k, 1) - \log C_1 = \phi_{ik}
\end{equation}

where $\lambda_k$ is the variational mean for mixture component $k$. Note that this step effectively amounts to computing the log-likelihood of sample $i$ under each component.

Now, for the global latent variables $\mu_k$, we have the following complete conditional distribution:

\begin{align}
p(\mu_k | x, z) &= \frac{1}{C_2} p(x | \mu_k, z) p(\mu_k) \ 
&= \frac{1}{C_2} \mathcal{N}(x | \mu_k, \sigma^2_0) \mathcal{N}(0, 1) \ 
&= \mathcal{N}(\widetilde{\mu}_k, \widetilde{\sigma^2})
\end{align}

where

\begin{equation}
\widetilde{\mu}_k = \frac{1}{n+1} \sum\limits_{i=1}^n \phi_{ik} x_i,~~~~\widetilde{\sigma^2} = \frac{1}{n+1}
\end{equation}

Suppose we sample data point $x_i$ on an iteration of SVI. The locally-optimized ELBO will occur when we set the pseudo-class assignment of $x_i$ to be $k^{*} = \text{argmax}_k \phi_{ik}$. The global update will then be

\begin{equation}
\lambda^{(t)}_{k^\star} = \frac{n}{n + 1} x_i
\end{equation}

We iteratively repeat this process, randomly sampling a data point at each iteration.

## Gaussian exponential family

We can put the Gaussian in the form of the exponential family. Assume we have a dataset $x = \{x_1, \dots, x_n\}$ that is assumed to have been drawn from a Gaussian with mean $\mu$ and variance $\sigma^2$. The likelihood is then:

\begin{align}
p(X | \mu, \sigma^2) &= \prod\limits_{i=1}^n \frac{1}{\sqrt{2\pi \sigma}} \exp\left\{-\frac{1}{2\sigma^2} (x_i - \mu)^2\right\} \\
&= \left(\frac{1}{\sqrt{2\pi \sigma}}\right)^n \exp\left\{-\frac{1}{2\sigma^2} \sum\limits_{i=1}^n (x_i - \mu)^2\right\} \\
&= \exp\left\{\log \left((2\pi \sigma)^{-n/2}\right) - \frac{1}{2\sigma^2} \sum\limits_{i=1}^n x_i^2 + \frac{\mu}{\sigma^2} \sum\limits_{i=1}^n x_i - \frac{n}{2\sigma^2} \mu^2) \right\} \\ 
&= \underbrace{2\pi^{-n/2}}_{h(x)} \exp\left\{ \underbrace{\begin{bmatrix} -\frac{1}{2\sigma^2} \\ \frac{\mu}{\sigma^2} \end{bmatrix}^\top}_{\eta(\theta)} \underbrace{\begin{bmatrix} \sum\limits_{i=1}^n x_i^2 \\ \sum\limits_{i=1}^n x_i \end{bmatrix}}_{t(x)} + \underbrace{\frac{n \mu^2}{2\sigma^2} - \frac{n}{2} \log \sigma)}_{a(\theta)} \right\}
\end{align}

# Implementation

In [ ]:
import numpy    as np
import pandas   as pd
import seaborn  as sns

from   scipy.stats import norm
import matplotlib.pyplot as plt

In [ ]:
def plot_data(x, means, zs, n_samples, jitter_sigma=None, colors=None):
    '''
    Plots (i) the data points with some vertical jitter to make visualization easier;
          (ii) the inferred gaussians.
    '''
    if jitter_sigma is None:
        jitter_sigma = 0.01
    jitter = np.random.normal(scale=jitter_sigma, size=n_samples)

    clrs  = [colors[z] for z in zs]
    k_arr = np.unique(zs) # [0 1 ... K-1]
    nk    = len(k_arr)

    fig   = plt.figure(figsize=(8,6))
    ax    = fig.subplots(2,1)
    fig.suptitle('Gaussian mixture model')

    xs = np.linspace(np.min(x), np.max(x), 1000)
    for z in range(nk):
        ys = norm.pdf(xs, loc=means[k_arr[z]], scale=sigma2)
        ax[0].plot(xs, ys, color=colors[z])

    ax[1].scatter(x, jitter, c=clrs)

    plt.show()

In [ ]:
# Generate the data

# Number of Gaussians in the mixture model
K       = 3

# True mean of the Gaussians
mu_true = np.array([-5, -1, 3])

# True variance of the Gaussians
sigma2  = 1

# Number of samples in the dataset
n       = 1000

# Draw 'n' random mapping to the considered K clusters
z_true  = np.random.choice(np.arange(3), replace=True, size=n)

# Draw 'n' random samples using the previously generated mapping to clusters
X       = np.random.normal(loc=mu_true[z_true], scale=np.sqrt(sigma2))

# Number of iterations for SVI optimization
n_iter  = 10000

# Parameters to adjust the step size during optimization.
# At iteration t: set_size = (t+tau)^{-kappa}

# Parameter that decreases the weight of the previous step size, when 
# computing the new step size
tau     = 1

# The forgetting rate
kappa   = 0.95

In [ ]:
# Initialize the parameters of the model

phi      = np.random.dirichlet(np.random.random(K), n) # local parameters
row_sums = phi.sum(axis=1)
phi      = phi / row_sums[:, np.newaxis]
_lambda  = np.random.normal(size=K)
mu_tilde = np.zeros(K)

In [ ]:
for iter_idx in range(n_iter):

    # Select a random ID of an observation
    data_idx = np.random.choice(np.arange(n))

    # Pick the observation whose ID was randomly selected
    Xi = X[data_idx]

    # Update the learning rate 'rho'
    rho = (iter_idx + 1 + tau)**(-kappa)

    # Update the local latent variables 'phi'
    for k in range(K):

        # Compute the local latent variables 'phi_ik'.
        # 'phi_ik' corresponds to the likelihood of 'x_i' belong to gaussian 'k'
        phi[data_idx, k] = norm.pdf(Xi, _lambda[k], sigma2)

    # Normalize the local latent variables 'phi_ik'
    row_sums = phi.sum(axis=1)
    phi      = phi / row_sums[:, np.newaxis]

    # Find the maximum of local latent variables: phi_i = MAX_k(phi_ik)
    max_idx = np.argmax(phi[data_idx, :])

    # Update the global latent variables '_lambda'
    _lambda[max_idx] = (1 - rho) * _lambda[max_idx] + rho * n / (n + 1) *  Xi

    print(_lambda)

In [ ]:
print('True means:     ', end="")
for m in range(K):
    print(f'{mu_true[m]:.4f}', end=" ")

print('\nInferred means: ', end="")
for m in range(K):
    print(f'{_lambda[m]:.4f}', end=" ")

In [ ]:
colors = ['g', 'r', 'y']

plot_data(X, means=_lambda, zs=z_true, n_samples=n, jitter_sigma=0.5, colors=colors)

Although the estimated means are not very precise, they really approximate the true Gaussian means. The method is highly scalable because it makes very few calculations.